## DPO implicit reward and measurement of pairwise ranking accuracy

The goal of this notebook is to sample 500 random triplets from `datasets/jdar_triplet_extracted_on_cuad_and_cold_cases/dpo_dataset_construction/held_out_pairs.jsonl` and compute the DPO implicit reward on the unseen triplets that were never used to train the model, providing a meaningful measure to compute the pairwise ranking accuracy and measure it.

In [1]:
%%capture
# Kept from the successful SFT notebook so SFT and DPO use one compatible runtime.
!pip3 install accelerate
!pip install -U "transformers>=5.10.4"
!pip install -U bitsandbytes>=0.46.1

In [2]:
import json
import os
from pathlib import Path

import torch
import transformers
from accelerate import PartialState

In [3]:
print(f"transformers={transformers.__version__}")

print(f"torch={torch.__version__}")

transformers=5.14.1
torch=2.10.0+cu128


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [5]:
SFT_PATH = "/kaggle/input/models/adrinorosario/llama-3-9b-sft-and-dpo-adapter-and-model/transformers/default/1/jdar_sft/jdar_sft/checkpoint-120"
DPO_PATH = "/kaggle/input/models/adrinorosario/llama-3-9b-sft-and-dpo-adapter-and-model/transformers/default/1/jdar_dpo_adapter/jdar_dpo_adapter"

In [6]:
print("Loading SFT Model...")
tokenizer_sft = AutoTokenizer.from_pretrained(SFT_PATH)
model_sft = AutoModelForCausalLM.from_pretrained(
    SFT_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

Loading SFT Model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

In [8]:
print("Loading DPO Model...")
tokenizer_dpo = AutoTokenizer.from_pretrained(DPO_PATH)
model_dpo = AutoModelForCausalLM.from_pretrained(
    DPO_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

Loading DPO Model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

In [9]:
import random

def load_and_sample_heldout_set(json_path, n):
    if n > 0:
        # read all the lines of the file
        with open(json_path, "r", encoding="UTF-8") as file:
            # parase each line as a json object
            data = [json.loads(line) for line in file]

        # randomly sample 500 json objects
        sample_size = min(n, len(data))
        sampled_data = random.sample(data, sample_size)

        cleaned_sampled_data = []
        for items in sampled_data:
            cleaned_sampled_data.append(
                (items['prompt_anchor'],
                items['chosen'],
                items['rejected'])
            )

        return cleaned_sampled_data
        
    else:
        print("Cannot sample 0 or negative number of samples")

In [10]:
heldout_500_samples = load_and_sample_heldout_set("/kaggle/input/datasets/adrinorosario/rejected-held-out-pairs/held_out_pairs.jsonl", 500)

len(heldout_500_samples)

500

In [11]:
model_sft.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=1024, bias=False)
            (lora_dropout): ModuleDict(
      

In [12]:
model_dpo.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=4096, out_features=1024, bias=False)
            (lora_dropout): ModuleDict(
      

In [13]:
def build_sequences(tokenizer, anchor, response):
    """Reproduces the dpo run's formatting exactly, for a single (anchor, response) pair."""
    eos = tokenizer.eos_token or ""
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": anchor}],
        tokenize=False,
        add_generation_prompt=True,
        reasoning_effort="high",
    )
    response_text = response if (not eos or response.endswith(eos)) else response + eos
    full_text = prompt_text + response_text
    return prompt_text, full_text

In [14]:
def sequence_log_probabilities(
    model, tokenizer,
    prompt_text, full_text,
    device
):
    # derive the input ids for both the prompt and full text
    prompt_ids = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False
    ).input_ids.to(device)

    full_ids = tokenizer(
        full_text,
        return_tensors="pt",
        add_special_tokens=False
    ).input_ids.to(device)
    
    labels = full_ids.clone()
    labels[:, :prompt_ids.shape[1]] = -100

    # compute the forward pass
    with torch.no_grad():
        out = model(full_ids, labels=labels)

    # compute the logprobs
    n_response_tokens = (labels != 100).sum().item()
    raw_logp = -out.loss.item() * n_response_tokens
    norm_logp = -out.loss.item()

    return raw_logp, norm_logp

In [15]:
results = []
beta = 0.1 # matching the the KL penalty from DPO config

In [16]:
device = "cuda"

In [17]:
from tqdm import tqdm

for item in tqdm(heldout_500_samples, desc="Evaluating DPO samples"):
    anchor, chosen, rejected = item

    #apply the formatting
    prompt_text, chosen_full_text = build_sequences(tokenizer_dpo, anchor, chosen)
    _, rejected_full_text = build_sequences(tokenizer_dpo, anchor, rejected)

    # score chosen under both models
    policy_chosen_raw, policy_chosen_norm = sequence_log_probabilities(
        model_dpo, tokenizer_dpo, prompt_text, chosen_full_text, device
    )
    ref_chosen_raw, ref_chosen_norm = sequence_log_probabilities(
        model_sft, tokenizer_sft, prompt_text, chosen_full_text, device
    )

    # score rejected under both models
    policy_rejected_raw, policy_rejected_norm = sequence_log_probabilities(
        model_dpo, tokenizer_dpo, prompt_text, rejected_full_text, device
    )
    ref_rejected_raw, ref_rejected_norm = sequence_log_probabilities(
        model_sft, tokenizer_sft, prompt_text, rejected_full_text, device
    )

    # implicit rewards
    r_chosen_raw = beta * (policy_chosen_raw - ref_chosen_raw)
    r_rejected_raw = beta * (policy_rejected_raw - ref_rejected_raw)
    r_chosen_norm = beta * (policy_chosen_norm - ref_chosen_norm)
    r_rejected_norm = beta * (policy_rejected_norm - ref_rejected_norm)

    results.append({
        "anchor": anchor,
        "r_chosen_raw": r_chosen_raw, "r_rejected_raw": r_rejected_raw,
        "r_chosen_norm": r_chosen_norm, "r_rejected_norm": r_rejected_norm,
        "correct_raw": r_chosen_raw > r_rejected_raw,
        "correct_norm": r_chosen_norm > r_rejected_norm,
    })

Evaluating DPO samples: 100%|██████████| 500/500 [1:00:24<00:00,  7.25s/it]


In [ ]:
# def batched_sequence_log_probs(model, tokenizer, prompt_texts, full_texts, device, batch_size=8):
#     """
#     Computes raw and length-normalized log P(continuation | prompt) for a batch of
#     (prompt, full_text) pairs, where full_text = prompt + continuation.

#     Returns: (raw_logprobs, norm_logprobs) as lists of floats, same order as input.
#     """
#     model.eval()
#     raw_logprobs, norm_logprobs = [], []

#     # Right-padding is fine for causal LMs as long as we mask out pad positions
#     # when summing log-probs (their logits are never used downstream here).
#     if tokenizer.padding_side != "right":
#         tokenizer.padding_side = "right"
#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token

#     for i in tqdm(range(0, len(full_texts), batch_size), desc="Scoring batches", leave=False):
#         batch_prompts = prompt_texts[i:i + batch_size]
#         batch_full = full_texts[i:i + batch_size]

#         # Tokenize prompts alone (no padding) just to get each prompt's token length,
#         # so we know where the continuation starts in the full sequence.
#         prompt_lens = [
#             len(tokenizer(p, add_special_tokens=True)["input_ids"])
#             for p in batch_prompts
#         ]

#         enc = tokenizer(
#             batch_full,
#             return_tensors="pt",
#             padding=True,
#             truncation=True,
#             add_special_tokens=True,
#         ).to(device)

#         input_ids = enc["input_ids"]
#         attention_mask = enc["attention_mask"]

#         with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
#             logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

#         # Standard causal LM shift: logits[:, t] predicts input_ids[:, t+1]
#         shift_logits = logits[:, :-1, :]
#         shift_labels = input_ids[:, 1:]
#         shift_attn = attention_mask[:, 1:]

#         log_probs = torch.log_softmax(shift_logits.float(), dim=-1)
#         token_log_probs = torch.gather(
#             log_probs, dim=2, index=shift_labels.unsqueeze(-1)
#         ).squeeze(-1)  # (batch, seq_len-1)

#         # Build a mask that's 1 only for continuation tokens that aren't padding.
#         # A position t in shift_labels corresponds to original token index t+1.
#         seq_len = shift_labels.size(1)
#         positions = torch.arange(seq_len, device=device).unsqueeze(0)  # (1, seq_len)
#         prompt_len_tensor = torch.tensor(prompt_lens, device=device).unsqueeze(1)  # (batch, 1)
#         # token at shifted position t is original index t+1; keep it if
#         # original index >= prompt_len (i.e. part of continuation)
#         continuation_mask = (positions + 1) >= prompt_len_tensor
#         mask = continuation_mask & shift_attn.bool()

#         masked_log_probs = token_log_probs * mask
#         raw_sum = masked_log_probs.sum(dim=1)                    # (batch,)
#         token_counts = mask.sum(dim=1).clamp(min=1)              # avoid div-by-zero
#         norm_sum = raw_sum / token_counts

#         raw_logprobs.extend(raw_sum.tolist())
#         norm_logprobs.extend(norm_sum.tolist())

#     return raw_logprobs, norm_logprobs

In [ ]:
# from tqdm import tqdm

# # Build all sequences up front
# anchors, prompts, chosen_texts, rejected_texts = [], [], [], []
# for anchor, chosen, rejected in heldout_500_samples:
#     prompt_text, chosen_full_text = build_sequences(tokenizer_dpo, anchor, chosen)
#     _, rejected_full_text = build_sequences(tokenizer_dpo, anchor, rejected)
#     anchors.append(anchor)
#     prompts.append(prompt_text)
#     chosen_texts.append(chosen_full_text)
#     rejected_texts.append(rejected_full_text)

# BATCH_SIZE = 8  # tune up/down based on T4 memory (try 16 if it fits)

# policy_chosen_raw, policy_chosen_norm = batched_sequence_log_probs(
#     model_dpo, tokenizer_dpo, prompts, chosen_texts, device, BATCH_SIZE
# )
# ref_chosen_raw, ref_chosen_norm = batched_sequence_log_probs(
#     model_sft, tokenizer_sft, prompts, chosen_texts, device, BATCH_SIZE
# )
# policy_rejected_raw, policy_rejected_norm = batched_sequence_log_probs(
#     model_dpo, tokenizer_dpo, prompts, rejected_texts, device, BATCH_SIZE
# )
# ref_rejected_raw, ref_rejected_norm = batched_sequence_log_probs(
#     model_sft, tokenizer_sft, prompts, rejected_texts, device, BATCH_SIZE
# )

# results = []
# for idx in range(len(anchors)):
#     r_chosen_raw = beta * (policy_chosen_raw[idx] - ref_chosen_raw[idx])
#     r_rejected_raw = beta * (policy_rejected_raw[idx] - ref_rejected_raw[idx])
#     r_chosen_norm = beta * (policy_chosen_norm[idx] - ref_chosen_norm[idx])
#     r_rejected_norm = beta * (policy_rejected_norm[idx] - ref_rejected_norm[idx])
#     results.append({
#         "anchor": anchors[idx],
#         "r_chosen_raw": r_chosen_raw, "r_rejected_raw": r_rejected_raw,
#         "r_chosen_norm": r_chosen_norm, "r_rejected_norm": r_rejected_norm,
#         "correct_raw": r_chosen_raw > r_rejected_raw,
#         "correct_norm": r_chosen_norm > r_rejected_norm,
#     })

In [19]:
output_path = "/kaggle/working/evaluation_results.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"File successfully saved and visible in Kaggle files sidebar: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.2f} KB")

File successfully saved and visible in Kaggle files sidebar: /kaggle/working/evaluation_results.json
File size: 273.22 KB


In [21]:
print(next(model_dpo.parameters()).device)
print(next(model_sft.parameters()).device)
print(device)

cuda:0
cuda:0
cuda
